# Parametric strength (no-context) per item

**Question:** is the capitals-concentrated drop really about *capitals*, or about
answer length / how strongly the model already knows the fact?

For every item, score the parametric and counterfactual answers **with no context**
(same prompt the Layer-0 filter uses: question + `The answer is:` prefill, chat template
on). That gives each item a continuous "how well does the model know this" number.
One forward pass per answer, so each model takes a couple of minutes.

Scored for: the **base model** (what pretraining left it with) and each run's
**peak** checkpoint (seed 0).

Output goes to Drive (persistent), one JSON per model. Download them into
`results/parametric-strength/` for the local regression.

In [1]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
!git log --oneline -3

!pip install -q peft accelerate

from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/CPI-Analysis-Results/parametric-strength"   # persistent
os.makedirs(OUT_DIR, exist_ok=True)
assert os.access(OUT_DIR, os.W_OK), "Drive output dir not writable"
print("writing to", OUT_DIR)

Mounted at /content/drive
writing to /content/drive/MyDrive/CPI-Analysis-Results/parametric-strength


In [2]:
import json, yaml, torch, sys, shutil, time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

sys.path.insert(0, "src/evaluation")
import eval as cpi_eval

cfg = yaml.safe_load(open("src/mechanistic-analysis/checkpoints.yaml"))
tok = AutoTokenizer.from_pretrained(cfg["base_model"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Llama-2-7b-hf ships no chat template; without this, eval.py silently falls back to a
# raw prompt the SFT models were never trained on. Must match run_sft.py's format.
tok.chat_template = (
    "{{ bos_token }} "
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ '<|system|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ '<|user|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|assistant|>\n' + message['content'].strip() + eos_token + '\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}"
)

# sanity: prompt format must match what the stored evals used, byte for byte
ref = json.load(open("results/cpi-results/tulu-results/checkpoint_step3150_metrics.json", encoding="utf-8"))
row = next(r for r in ref["per_item"] if r["item_id"] == "cap_0013")
ours = tok.decode(cpi_eval.build_score_prompt_ids(tok, row["question"], row["context"], True),
                  skip_special_tokens=False)
assert ours == row["score_prompt_text"], f"PROMPT MISMATCH\nours  : {ours!r}\nstored: {row['score_prompt_text']!r}"
assert len(tok(" Kabul", add_special_tokens=False).input_ids) == 2, "double-space bug is back"
print("prompt + tokenisation verified")

items = cpi_eval.load_items("dataset/conflict_eval_unified.json")
base_model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"], torch_dtype=torch.bfloat16, device_map="cuda").eval()
print(len(items), "items; base model loaded")

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

412 items; base model loaded


In [3]:
STAGE = "/content/_ckpt"

def load(adapter_path):
    """Base model with the given LoRA adapter merged in (None = plain base model)."""
    global base_model
    if isinstance(base_model, PeftModel):
        base_model = base_model.unload()
    if adapter_path is None:
        return base_model.eval()
    shutil.rmtree(STAGE, ignore_errors=True)
    shutil.copytree(adapter_path, STAGE)          # local copy: Drive reads are slow
    base_model = PeftModel.from_pretrained(base_model, STAGE)
    return base_model.eval()

def score_no_context(model):
    """Per item: log P(answer | question, NO context), for parametric and counterfactual."""
    rows = []
    for it in items:
        pid = cpi_eval.build_score_prompt_ids(tok, it["question"], None, True)
        par_lp, n_par = cpi_eval.score_answer_logprob(model, tok, pid, it["parametric_answer"])
        cf_lp, n_cf = cpi_eval.score_answer_logprob(model, tok, pid, it["counterfactual_answer"])
        rows.append({
            "item_id": it["item_id"], "source": it["source"],
            "n_par": n_par, "n_cf": n_cf,
            "par_lp_total": round(par_lp, 4), "cf_lp_total": round(cf_lp, 4),
            "par_lp_pt": round(par_lp / max(n_par, 1), 4),       # per-token, same as the filter
            "cf_lp_pt": round(cf_lp / max(n_cf, 1), 4),
        })
    return rows

run_dir = lambda run: cfg["runs"][run]["drive_checkpoint_dir"]
MODELS = {
    "base":             None,
    "tulu_seed0_peak":   f"{run_dir('tulu')}/{cfg['runs']['tulu']['phases']['peak']}",
    "alpaca_seed0_peak": f"{run_dir('alpaca')}/{cfg['runs']['alpaca']['phases']['peak']}",
}
print(MODELS)

{'base': None, 'tulu_seed0_peak': '/content/drive/MyDrive/Checkpoints - TULU - CPI- Analysis/checkpoints/checkpoint-500', 'alpaca_seed0_peak': '/content/drive/MyDrive/Checkpoints - Alpaca - CPI- Analysis/Alpaca_Checkpoints/checkpoints/checkpoint-100'}


In [5]:
!pip uninstall -q -y torchao

In [6]:
# Full run. Resumable: a model whose JSON is already on Drive is skipped.
STORED_PEAK = {"tulu_seed0_peak":   "results/cpi-results/tulu-results/checkpoint_step500_metrics.json",
               "alpaca_seed0_peak": "results/cpi-results/alpaca-results/checkpoint_step100_metrics.json"}

for name, adapter in MODELS.items():
    out_file = f"{OUT_DIR}/{name}.json"
    if os.path.exists(out_file):
        print(f"{name}: already on Drive, skip")
        continue
    t0 = time.time()
    rows = score_no_context(load(adapter))

    # Check: every item the stored eval marked "passed" at this checkpoint must have
    # par >= cf with no context (the filter requires par to beat cf and 5 distractors).
    if name in STORED_PEAK:
        stored = {r["item_id"]: r["passed"] for r in json.load(open(STORED_PEAK[name], encoding="utf-8"))["per_item"]}
        bad = [r["item_id"] for r in rows if stored.get(r["item_id"]) and r["par_lp_pt"] < r["cf_lp_pt"]]
        print(f"  {name}: passed-but-par<cf = {len(bad)} (want 0) {bad[:5]}")

    json.dump({"model": name, "adapter": adapter, "per_item": rows}, open(out_file, "w"), indent=1)
    print(f"{name}: {len(rows)} items in {time.time() - t0:.0f}s -> {out_file}")

shutil.rmtree(STAGE, ignore_errors=True)

base: already on Drive, skip
  tulu_seed0_peak: passed-but-par<cf = 0 (want 0) []
tulu_seed0_peak: 412 items in 76s -> /content/drive/MyDrive/CPI-Analysis-Results/parametric-strength/tulu_seed0_peak.json
  alpaca_seed0_peak: passed-but-par<cf = 0 (want 0) []
alpaca_seed0_peak: 412 items in 102s -> /content/drive/MyDrive/CPI-Analysis-Results/parametric-strength/alpaca_seed0_peak.json


In [7]:
# Small sanity check first: 5 items on the base model, eyeball the numbers.
_all = items
items = _all[:5]
for r in score_no_context(load(None)):
    print(r)
items = _all

{'item_id': 'cap_0000', 'source': 'country_capitals', 'n_par': 2, 'n_cf': 2, 'par_lp_total': -0.2613, 'cf_lp_total': -7.0833, 'par_lp_pt': -0.1306, 'cf_lp_pt': -3.5416}
{'item_id': 'cap_0001', 'source': 'country_capitals', 'n_par': 2, 'n_cf': 4, 'par_lp_total': -0.8611, 'cf_lp_total': -6.1774, 'par_lp_pt': -0.4306, 'cf_lp_pt': -1.5443}
{'item_id': 'cap_0002', 'source': 'country_capitals', 'n_par': 3, 'n_cf': 2, 'par_lp_total': -0.4445, 'cf_lp_total': -7.1003, 'par_lp_pt': -0.1482, 'cf_lp_pt': -3.5502}
{'item_id': 'cap_0003', 'source': 'country_capitals', 'n_par': 6, 'n_cf': 2, 'par_lp_total': -1.0018, 'cf_lp_total': -8.866, 'par_lp_pt': -0.167, 'cf_lp_pt': -4.433}
{'item_id': 'cap_0004', 'source': 'country_capitals', 'n_par': 2, 'n_cf': 3, 'par_lp_total': -0.6217, 'cf_lp_total': -5.1942, 'par_lp_pt': -0.3109, 'cf_lp_pt': -1.7314}
